In [7]:
import pandas as pd
import numpy as np
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
import pyarrow.parquet as pq
import gc

# --- 1. CẤU HÌNH ---
train_path = '/kaggle/input/notebooks/luminhanh/ba-model-prep-data/train_clean.parquet'
test_path = '/kaggle/input/notebooks/luminhanh/ba-model-prep-data/test_clean.parquet'
target_col = 'unit_sales'

# Khởi tạo mô hình - giữ khung Huber loss giúp xử lý outliers tốt
sgd = SGDRegressor(loss='huber', penalty='elasticnet', alpha=0.05, learning_rate='adaptive', eta0=0.0001)
scaler = StandardScaler()

# --- 2. TẠO RECENCY WEIGHTED ENCODING ---
print("Đang tính toán Weighted Encoding & Momentum từ dữ liệu mới...")
parquet_file = pq.ParquetFile(train_path)
df_sample = next(parquet_file.iter_batches(batch_size=10000000)).to_pandas()

# Trọng số ưu tiên tháng gần (Ecuador Retail thường biến động mạnh theo mùa)
df_sample['recency_weight'] = 1.0
df_sample.loc[df_sample['month'] == 7, 'recency_weight'] = 1.2
df_sample.loc[df_sample['month'] == 8, 'recency_weight'] = 1.5

# Tính Weighted Mean cho unit_sales
df_sample['weighted_sales'] = df_sample[target_col] * df_sample['recency_weight']

target_means = {}
for col in ['store_nbr', 'item_nbr', 'family', 'cluster', 'class']:
    group = df_sample.groupby(col, observed=True)
    target_means[col] = (group['weighted_sales'].sum() / group['recency_weight'].sum()).to_dict()

# Feature tương tác Promo x Family 
df_sample['promo_family'] = df_sample['onpromotion'].astype(int).astype(str) + "_" + df_sample['family'].astype(str)
group_pf = df_sample.groupby('promo_family', observed=True)
target_means['promo_family'] = (group_pf['weighted_sales'].sum() / group_pf['recency_weight'].sum()).to_dict()

global_mean = df_sample[target_col].mean()

# Tính Momentum cho Family (Xu hướng tăng trưởng của nhóm hàng)
early_mean = df_sample[df_sample['month'] < 6].groupby('family', observed=True)[target_col].mean()
recent_mean = df_sample[df_sample['month'] >= 6].groupby('family', observed=True)[target_col].mean()
momentum_family = (recent_mean / early_mean).replace([np.inf, -np.inf], 1.0).fillna(1.0).to_dict()

del df_sample
gc.collect()

# --- 3. HUẤN LUYỆN (BATCH TRAINING) ---
print("Bắt đầu huấn luyện với 61 features + Engineered features...")
feature_names = None

for i, batch in enumerate(parquet_file.iter_batches(batch_size=1000000)):
    chunk = batch.to_pandas()
    
    # --- Áp dụng Encoding & Momentum ---
    for col in ['store_nbr', 'item_nbr', 'family', 'cluster', 'class']:
        chunk[f'{col}_enc'] = chunk[col].map(target_means.get(col, {})).fillna(global_mean)
        
    chunk['family_momentum'] = chunk['family'].map(momentum_family).fillna(1.0)
    chunk['promo_family_enc'] = (chunk['onpromotion'].astype(int).astype(str) + "_" + 
                                 chunk['family'].astype(str)).map(target_means['promo_family']).fillna(global_mean)

    # Chọn tất cả cột số (bao gồm 61 cột gốc và các cột enc mới)
    X_batch = chunk.select_dtypes(include=[np.number]).drop(columns=[target_col, 'id'], errors='ignore')
    X_batch = X_batch.fillna(0).replace([np.inf, -np.inf], 0)
    
    if feature_names is None:
        feature_names = X_batch.columns.tolist()
    
    X_batch = X_batch[feature_names]
    scaler.partial_fit(X_batch)
    
    # --- Tính toán Sample Weight ---
    # Ưu tiên: Hàng dễ hỏng (perishable) + Tháng gần nhất (Recency Boost)
    recency_boost = chunk['month'].map({8: 1.3, 7: 1.1}).fillna(1.0)

    weights = (chunk['perishable'].astype(float) * 0.25 + 1) * recency_boost
    
    sgd.partial_fit(scaler.transform(X_batch), chunk[target_col].values, sample_weight=weights)
    
    if i % 5 == 0:
        print(f"Đã huấn luyện xong batch {i}")

# --- 4. DỰ BÁO ---
print("Đang dự báo trên tập Test...")
test_df = pd.read_parquet(test_path)

for col in ['store_nbr', 'item_nbr', 'family', 'cluster', 'class']:
    test_df[f'{col}_enc'] = test_df[col].map(target_means.get(col, {})).fillna(global_mean)

test_df['family_momentum'] = test_df['family'].map(momentum_family).fillna(1.0)
test_df['promo_family_enc'] = (test_df['onpromotion'].astype(int).astype(str) + "_" + 
                               test_df['family'].astype(str)).map(target_means['promo_family']).fillna(global_mean)

X_test_final = test_df.select_dtypes(include=[np.number]).reindex(columns=feature_names).fillna(0).replace([np.inf, -np.inf], 0)

preds = sgd.predict(scaler.transform(X_test_final))
y_preds = np.expm1(np.clip(preds, 0, 6.4)) * 0.99

submission = pd.DataFrame({'id': test_df['id'], 'unit_sales': y_preds})
submission.to_csv('submission_optimized_61f.csv', index=False)
print("Hoàn tất! File nộp đã được lưu.")

Đang tính toán Weighted Encoding & Momentum từ dữ liệu mới...


/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:52: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:731: RuntimeWarning: invalid value encountered in scalar divide
  the_mean = the_sum / count if count > 0 else np.nan


Bắt đầu huấn luyện với 61 features + Engineered features...
Đã huấn luyện xong batch 0
Đã huấn luyện xong batch 5
Đã huấn luyện xong batch 10
Đã huấn luyện xong batch 15
Đã huấn luyện xong batch 20
Đã huấn luyện xong batch 25
Đã huấn luyện xong batch 30
Đã huấn luyện xong batch 35
Đã huấn luyện xong batch 40
Đã huấn luyện xong batch 45
Đã huấn luyện xong batch 50
Đã huấn luyện xong batch 55
Đã huấn luyện xong batch 60
Đã huấn luyện xong batch 65
Đang dự báo trên tập Test...
Hoàn tất! File nộp đã được lưu.


In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
import pyarrow.parquet as pq
import gc

# --- 1. CẤU HÌNH ---
train_path = '/kaggle/input/notebooks/luminhanh/ba-model-prep-data/train_clean.parquet'
test_path = '/kaggle/input/notebooks/luminhanh/ba-model-prep-data/test_clean.parquet'
target_col = 'unit_sales'

# Khởi tạo mô hình - giữ khung Huber loss giúp xử lý outliers tốt
sgd = SGDRegressor(
    loss='huber',
    penalty='elasticnet',
    alpha=0.05,
    learning_rate='adaptive',
    eta0=0.0001,
    random_state=42
)

scaler = StandardScaler()

# --- 2. TẠO RECENCY WEIGHTED ENCODING ---
print("Đang tính toán Weighted Encoding & Momentum từ dữ liệu mới...")
parquet_file = pq.ParquetFile(train_path)
df_sample = next(parquet_file.iter_batches(batch_size=10000000)).to_pandas()

# QUAN TRỌNG: tránh overflow
df_sample[target_col] = df_sample[target_col].astype('float64')

# Trọng số ưu tiên tháng gần
df_sample['recency_weight'] = 1.0
df_sample.loc[df_sample['month'] == 7, 'recency_weight'] = 1.2
df_sample.loc[df_sample['month'] == 8, 'recency_weight'] = 1.5

# Tính Weighted Mean cho unit_sales
df_sample['weighted_sales'] = (
    df_sample[target_col] * df_sample['recency_weight']
)

target_means = {}
for col in ['store_nbr', 'item_nbr', 'family', 'cluster', 'class']:
    group = df_sample.groupby(col, observed=True)
    target_means[col] = (
        group['weighted_sales'].sum() /
        group['recency_weight'].sum()
    ).to_dict()

# Feature tương tác Promo x Family
df_sample['promo_family'] = (
    df_sample['onpromotion'].astype(int).astype(str)
    + "_"
    + df_sample['family'].astype(str)
)

group_pf = df_sample.groupby('promo_family', observed=True)
target_means['promo_family'] = (
    group_pf['weighted_sales'].sum() /
    group_pf['recency_weight'].sum()
).to_dict()

global_mean = df_sample[target_col].mean()

# Tính Momentum cho Family
early_mean = (
    df_sample[df_sample['month'] < 6]
    .groupby('family', observed=True)[target_col]
    .mean()
)

recent_mean = (
    df_sample[df_sample['month'] >= 6]
    .groupby('family', observed=True)[target_col]
    .mean()
)

momentum_family = (
    recent_mean / early_mean
).replace([np.inf, -np.inf], 1.0).fillna(1.0).to_dict()

del df_sample
gc.collect()

# --- 3. HUẤN LUYỆN (BATCH TRAINING) ---
print("Bắt đầu huấn luyện với 61 features + Engineered features...")
feature_names = None

for i, batch in enumerate(parquet_file.iter_batches(batch_size=1000000)):
    chunk = batch.to_pandas()

    # QUAN TRỌNG: tránh overflow
    chunk[target_col] = chunk[target_col].astype('float64')

    # --- Áp dụng Encoding & Momentum ---
    for col in ['store_nbr', 'item_nbr', 'family', 'cluster', 'class']:
        chunk[f'{col}_enc'] = (
            chunk[col]
            .map(target_means[col])
            .fillna(global_mean)
            .astype('float64')
        )

    chunk['family_momentum'] = (
        chunk['family']
        .map(momentum_family)
        .fillna(1.0)
        .astype('float64')
    )

    chunk['promo_family_enc'] = (
        chunk['onpromotion'].astype(int).astype(str)
        + "_"
        + chunk['family'].astype(str)
    ).map(target_means['promo_family']).fillna(global_mean).astype('float64')

    # Chọn tất cả cột số
    X_batch = (
        chunk.select_dtypes(include=[np.number])
        .drop(columns=[target_col, 'id'], errors='ignore')
        .astype('float64')
        .replace([np.inf, -np.inf], 0)
        .fillna(0)
    )

    if feature_names is None:
        feature_names = X_batch.columns.tolist()

    X_batch = X_batch.reindex(columns=feature_names, fill_value=0)

    scaler.partial_fit(X_batch)

    # --- Tính toán Sample Weight ---
    recency_boost = (
        chunk['month']
        .map({8: 1.3, 7: 1.1})
        .fillna(1.0)
        .astype('float64')
    )

    weights = (
        (chunk['perishable'].astype('float64') * 0.25 + 1.0)
        * recency_boost
    )

    sgd.partial_fit(
        scaler.transform(X_batch),
        chunk[target_col].values,
        sample_weight=weights.values
    )

    if i % 5 == 0:
        print(f"Đã huấn luyện xong batch {i}")

    del chunk, X_batch, weights
    gc.collect()

# --- 4. DỰ BÁO ---
print("Đang dự báo trên tập Test...")
test_df = pd.read_parquet(test_path)

for col in ['store_nbr', 'item_nbr', 'family', 'cluster', 'class']:
    test_df[f'{col}_enc'] = (
        test_df[col]
        .map(target_means[col])
        .fillna(global_mean)
        .astype('float64')
    )

test_df['family_momentum'] = (
    test_df['family']
    .map(momentum_family)
    .fillna(1.0)
    .astype('float64')
)

test_df['promo_family_enc'] = (
    test_df['onpromotion'].astype(int).astype(str)
    + "_"
    + test_df['family'].astype(str)
).map(target_means['promo_family']).fillna(global_mean).astype('float64')

X_test_final = (
    test_df.select_dtypes(include=[np.number])
    .drop(columns=['id'], errors='ignore')
    .reindex(columns=feature_names, fill_value=0)
    .astype('float64')
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

preds = sgd.predict(scaler.transform(X_test_final))
y_preds = np.expm1(np.clip(preds, 0, 6.4)) * 0.99

submission = pd.DataFrame({
    'id': test_df['id'],
    'unit_sales': y_preds
})

submission.to_csv('submission_optimized_61f.csv', index=False)
print("Hoàn tất! File nộp đã được lưu.")

Đang tính toán Weighted Encoding & Momentum từ dữ liệu mới...
Bắt đầu huấn luyện với 61 features + Engineered features...
Đã huấn luyện xong batch 0
Đã huấn luyện xong batch 5
Đã huấn luyện xong batch 10
Đã huấn luyện xong batch 15
Đã huấn luyện xong batch 20
Đã huấn luyện xong batch 25
Đã huấn luyện xong batch 30
Đã huấn luyện xong batch 35
Đã huấn luyện xong batch 40
Đã huấn luyện xong batch 45
Đã huấn luyện xong batch 50
Đã huấn luyện xong batch 55
Đã huấn luyện xong batch 60
Đã huấn luyện xong batch 65
Đang dự báo trên tập Test...
Hoàn tất! File nộp đã được lưu.
